# 3-Hop Path Finder

This notebook demonstrates how to find 3-hop paths between two nodes through two intermediate categories using the Translator ecosystem.

**Workflow:**
1. Load Translator resources
2. Define start/end nodes and intermediate categories
3. Select predicates and APIs for each hop
4. Query each hop separately
5. Find intersecting intermediate nodes to assemble complete paths

In [ ]:
from TCT import name_resolver, translator_query, TCT
from TCT.translator_resources import TranslatorResources

resources = TranslatorResources.load()
print(f"Number of APIs: {len(resources.api_names)}")
print(f"MetaKG shape: {resources.meta_kg.shape}")

In [ ]:
# Define input parameters
start_node_name = 'imatinib'
start_node_id = TCT.get_curie(start_node_name)
start_node_ids = [start_node_id]
start_node_categories = ['biolink:Drug', 'biolink:SmallMolecule']
print(f"Start node: {start_node_name} -> {start_node_id}")

end_node_name = 'asthma'
end_node_id = TCT.get_curie(end_node_name)
end_node_ids = [end_node_id]
end_node_categories = ['biolink:Disease']
print(f"End node: {end_node_name} -> {end_node_id}")

# Intermediate categories for the two intermediate hops
intermediate1_categories = ['biolink:Gene', 'biolink:Protein']
intermediate2_categories = ['biolink:Cell']

In [ ]:
# Select predicates and APIs for all 3 hops

# Hop 1: start_node -> intermediate1
sele_predicates_1 = list(set(TCT.select_concept(
    sub_list=start_node_categories, obj_list=intermediate1_categories, metaKG=resources.meta_kg)))
sele_APIs_1 = TCT.select_API(
    sub_list=start_node_categories, obj_list=intermediate1_categories, metaKG=resources.meta_kg)
print(f"Hop 1: {len(sele_predicates_1)} predicates, {len(sele_APIs_1)} APIs")

# Hop 2: end_node -> intermediate2
sele_predicates_2 = list(set(TCT.select_concept(
    sub_list=end_node_categories, obj_list=intermediate2_categories, metaKG=resources.meta_kg)))
sele_APIs_2 = TCT.select_API(
    sub_list=end_node_categories, obj_list=intermediate2_categories, metaKG=resources.meta_kg)
print(f"Hop 2: {len(sele_predicates_2)} predicates, {len(sele_APIs_2)} APIs")

# Hop 3: intermediate1 <-> intermediate2
sele_predicates_3 = list(set(
    TCT.select_concept(sub_list=intermediate1_categories, obj_list=intermediate2_categories, metaKG=resources.meta_kg)
    + TCT.select_concept(sub_list=intermediate2_categories, obj_list=intermediate1_categories, metaKG=resources.meta_kg)))
sele_APIs_3 = TCT.select_API(
    sub_list=intermediate1_categories, obj_list=intermediate2_categories, metaKG=resources.meta_kg)
print(f"Hop 3: {len(sele_predicates_3)} predicates, {len(sele_APIs_3)} APIs")

In [ ]:
# Hop 1: start_node -> intermediate1
query_json1 = TCT.format_query_json(
    subject_ids=start_node_ids,
    object_ids=[],
    subject_categories=start_node_categories,
    object_categories=intermediate1_categories,
    predicates=sele_predicates_1)

result1 = translator_query.parallel_api_query(
    query_json=query_json1,
    select_APIs=list(sele_APIs_1),
    resources=resources,
    max_workers=len(sele_APIs_1))

result_parsed1 = TCT.parse_KG(result=result1)
print(f"Hop 1: {len(result1)} edges, {len(result_parsed1)} parsed entries")

In [ ]:
# Hop 2: end_node -> intermediate2
query_json2 = TCT.format_query_json(
    subject_ids=end_node_ids,
    object_ids=[],
    subject_categories=end_node_categories,
    object_categories=intermediate2_categories,
    predicates=sele_predicates_2)

result2 = translator_query.parallel_api_query(
    query_json=query_json2,
    select_APIs=list(sele_APIs_2),
    resources=resources,
    max_workers=len(sele_APIs_2))

result_parsed2 = TCT.parse_KG(result=result2)
print(f"Hop 2: {len(result2)} edges, {len(result_parsed2)} parsed entries")

In [ ]:
# Extract intermediate node sets from hops 1 & 2
intermediate1_ids = set()
for key in result_parsed1:
    entry = result_parsed1[key]
    if entry['subject'] in start_node_ids:
        intermediate1_ids.add(entry['object'])
    elif entry['object'] in start_node_ids:
        intermediate1_ids.add(entry['subject'])
print(f"Intermediate1 nodes (from hop 1): {len(intermediate1_ids)}")

intermediate2_ids = set()
for key in result_parsed2:
    entry = result_parsed2[key]
    if entry['subject'] in end_node_ids:
        intermediate2_ids.add(entry['object'])
    elif entry['object'] in end_node_ids:
        intermediate2_ids.add(entry['subject'])
print(f"Intermediate2 nodes (from hop 2): {len(intermediate2_ids)}")

In [ ]:
# Hop 3: query the smaller intermediate set against the other category
if len(intermediate1_ids) <= len(intermediate2_ids):
    query_ids = list(intermediate1_ids)
    query_categories = intermediate1_categories
    target_categories = intermediate2_categories
    print(f"Querying {len(query_ids)} intermediate1 nodes -> intermediate2")
else:
    query_ids = list(intermediate2_ids)
    query_categories = intermediate2_categories
    target_categories = intermediate1_categories
    print(f"Querying {len(query_ids)} intermediate2 nodes -> intermediate1")

query_json3 = TCT.format_query_json(
    subject_ids=query_ids,
    object_ids=[],
    subject_categories=query_categories,
    object_categories=target_categories,
    predicates=sele_predicates_3)

result3 = translator_query.parallel_api_query(
    query_json=query_json3,
    select_APIs=list(sele_APIs_3),
    resources=resources,
    max_workers=len(sele_APIs_3))

result_parsed3 = TCT.parse_KG(result=result3)
print(f"Hop 3: {len(result3)} edges, {len(result_parsed3)} parsed entries")

In [ ]:
# Assemble 3-hop paths from intersection
paths_found = set()

for key in result_parsed3:
    entry = result_parsed3[key]
    subj = entry['subject']
    obj = entry['object']

    if len(intermediate1_ids) <= len(intermediate2_ids):
        # queried intermediate1 -> intermediate2
        if subj in intermediate1_ids and obj in intermediate2_ids:
            paths_found.add(f"{start_node_name} > {subj} > {obj} > {end_node_name}")
        elif obj in intermediate1_ids and subj in intermediate2_ids:
            paths_found.add(f"{start_node_name} > {obj} > {subj} > {end_node_name}")
    else:
        # queried intermediate2 -> intermediate1
        if subj in intermediate2_ids and obj in intermediate1_ids:
            paths_found.add(f"{start_node_name} > {obj} > {subj} > {end_node_name}")
        elif obj in intermediate2_ids and subj in intermediate1_ids:
            paths_found.add(f"{start_node_name} > {subj} > {obj} > {end_node_name}")

print(f"Found {len(paths_found)} 3-hop paths")
for p in list(paths_found)[:20]:
    print(p)

In [ ]:
# Optional: visualize a selected path
from TCT import node_normalizer

if paths_found:
    sample_path = list(paths_found)[0]
    parts = sample_path.split(' > ')
    print(f"Sample path: {sample_path}")
    # Look up names for intermediate nodes
    for node_id in parts[1:-1]:
        try:
            names = node_normalizer.get_normalized_nodes([node_id])
            if names and node_id in names and names[node_id]:
                label = names[node_id].get('id', {}).get('label', node_id)
                print(f"  {node_id} -> {label}")
        except Exception:
            print(f"  {node_id} -> (name lookup failed)")
else:
    print("No 3-hop paths found to visualize")